# Páginas de inventario y QR — *De Cádiz a Nueva York*

Genera una página independiente por inventario (`inventarios/<año>.html`), con su facsímil, sus cifras y su QR, más una página de resultados. La presentación `inventarios/index.html` sirve de índice.

*Nota de uso de IA: tabla, código y textos preparados con Claude bajo la dirección de los autores, que fijan las ideas centrales y las conclusiones; IA con salvaguardas y transparencia.*

In [1]:
import pandas as pd, qrcode, html, os
from PIL import Image, ImageDraw, ImageFont
from datetime import date
BASE_URL = "https://jjr2025.github.io/De_Cadiz_a_Nueva_York/"
OUT = "inventarios"; os.makedirs(f"{OUT}/qr", exist_ok=True)

In [2]:
inv = pd.DataFrame([
 dict(id="1805", lugar="1805 · Cádiz", titulo="La partición: dos hijuelas",
      texto="Cinco años después de la muerte de Sebastián Martínez, sus dos hijas se reparten la pinacoteca en lotes de valor equivalente: Josefa, casada con Fernando Casado de Torres, y Catalina, casada con Francisco Viola.",
      img="img/1805.jpg", pie="«Partición convencional de los bienes quedados por muerte del Sr. D. Sebastián Martínez, Thesorero General que fue del Reyno, hecha entre sus dos hijas y herederas D.ª Josefa y D.ª Catalina Martínez», portada.",
      fuente="Archivo Histórico de Protocolos de Cádiz", tam="591 asientos", valor="295.420 reales", ver="Qué recibió cada hija"),
 dict(id="1828", lugar="1828 · Madrid", titulo="El inventario de Fernando Casado",
      texto="El marino e ingeniero recompone la colección: recupera buena parte de lo que había recibido Catalina y la amplía hasta casi mil obras.",
      img="img/1828.jpg", pie="«Catálogo y tasación de la pinacoteca de… Fernando Casado de Torres», portada.",
      fuente="Biblioteca Nacional de España, Mss/18630/40", tam="956 obras", valor="c. 1.048.000 reales", ver="Qué volvió a la familia y qué se añadió"),
 dict(id="1844", lugar="1844-1849 · Madrid", titulo="El avalúo de la Academia y las ofertas a Isabel II",
      texto="Celestino García Luz, yerno de Casado, ofrece la colección a la Corona en 1844, 1847 y 1849. La Academia de San Fernando la tasa. La reina no compra.",
      img="img/1844.jpg", pie="«Inventario y tasación de los cuadros que formaban la colección de don Celestino García Luz», carpetilla.",
      fuente="Real Academia de Bellas Artes de San Fernando, Archivo, sign. 1-36-6", tam="658 obras", valor="1.404.080 reales", ver="El juicio de los expertos"),
 dict(id="1867", lugar="27 de junio de 1867 · París", titulo="La venta del conde del Retamoso en el Hôtel Drouot",
      texto="El mismo año en que el marqués de Salamanca vende su galería en París, los herederos lo intentan con 51 lotes. El resultado es pobre: al menos 23 lotes, y 18 de los 19 Murillos, vuelven a aparecer en la partición de 1888.",
      img="img/1867.jpg", pie="Catálogo de la venta de la galería de M. P. Muñoz, conde del Retamoso, Hôtel Drouot, con anotaciones de precios a mano.",
      fuente="Lugt 29905 (Brill, Art Sales Catalogues Online)", tam="51 lotes, 19 de ellos Murillos", valor="—", ver="El intento de vender en París"),
 dict(id="1888", lugar="1885-1888 · Madrid", titulo="La partición de Josefa García-Luz",
      texto="Josefa muere en 1885 y la partición se cierra en 1888. Es el inventario que deja ver lo que no se vendió en París; en él aparece el Goya de Sebastián, tasado en 375 pesetas.",
      img="img/1888.jpg", pie="Testamentaría de Josefa García-Luz Casado, condesa del Retamoso: el legajo.",
      fuente="Archivo Histórico de Protocolos de Madrid, t. 28102", tam="261 asientos", valor="c. 160.000 pesetas", ver="Lo que no se vendió en París"),
 dict(id="1937", lugar="13 de mayo de 1937 · Tarancón", titulo="El acta de la Junta de Incautación",
      texto="Diez meses después del asesinato de la condesa viuda del Retamoso, la Junta registra lo que queda en el palacio de Tarancón. Entre los cuadros está el retrato de Sebastián por Inza, «rasgado en el centro».",
      img="img/1937.jpg", pie="Junta Central del Tesoro Artístico, Junta Delegada de Cuenca: acta n.º 1.",
      fuente="Actas 1 a 4 de la Junta Delegada de Cuenca y acta del Parque de Automóviles", tam="39 cuadros: 20 religiosos y 19 retratos", valor="—", ver="Lo que quedaba al final"),
])
inv["url"] = BASE_URL + "inventarios/" + inv["id"] + ".html"
inv

,id,lugar,titulo,texto,img,pie,fuente,tam,valor,ver,url
0,1805,1805 · Cádiz,La partición: dos hijuelas,Cinco años después de la muerte de Sebastián M...,img/1805.jpg,«Partición convencional de los bienes quedados...,Archivo Histórico de Protocolos de Cádiz,591 asientos,295.420 reales,Qué recibió cada hija,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
1,1828,1828 · Madrid,El inventario de Fernando Casado,El marino e ingeniero recompone la colección: ...,img/1828.jpg,«Catálogo y tasación de la pinacoteca de… Fern...,"Biblioteca Nacional de España, Mss/18630/40",956 obras,c. 1.048.000 reales,Qué volvió a la familia y qué se añadió,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
2,1844,1844-1849 · Madrid,El avalúo de la Academia y las ofertas a Isabe...,"Celestino García Luz, yerno de Casado, ofrece ...",img/1844.jpg,«Inventario y tasación de los cuadros que form...,"Real Academia de Bellas Artes de San Fernando,...",658 obras,1.404.080 reales,El juicio de los expertos,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
3,1867,27 de junio de 1867 · París,La venta del conde del Retamoso en el Hôtel Dr...,El mismo año en que el marqués de Salamanca ve...,img/1867.jpg,Catálogo de la venta de la galería de M. P. Mu...,"Lugt 29905 (Brill, Art Sales Catalogues Online)","51 lotes, 19 de ellos Murillos",—,El intento de vender en París,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
4,1888,1885-1888 · Madrid,La partición de Josefa García-Luz,Josefa muere en 1885 y la partición se cierra ...,img/1888.jpg,"Testamentaría de Josefa García-Luz Casado, con...","Archivo Histórico de Protocolos de Madrid, t. ...",261 asientos,c. 160.000 pesetas,Lo que no se vendió en París,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...
5,1937,13 de mayo de 1937 · Tarancón,El acta de la Junta de Incautación,Diez meses después del asesinato de la condesa...,img/1937.jpg,"Junta Central del Tesoro Artístico, Junta Dele...",Actas 1 a 4 de la Junta Delegada de Cuenca y a...,39 cuadros: 20 religiosos y 19 retratos,—,Lo que quedaba al final,https://jjr2025.github.io/De_Cadiz_a_Nueva_Yor...


In [3]:
CSS = """:root{--bg:#F6F3EC;--ink:#1E2329;--muted:#4A5560;--accent:#0B769F;--line:#d9d3c6;--card:#fffdf8}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#15191E;--ink:#F3EFE6;--muted:#B9C2CB;--accent:#7CC3DE;--line:#3A424B;--card:#1d2329}}
:root[data-theme="dark"]{--bg:#15191E;--ink:#F3EFE6;--muted:#B9C2CB;--accent:#7CC3DE;--line:#3A424B;--card:#1d2329}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:Georgia,serif}
main{max-width:900px;margin:auto;padding:28px 16px 48px}
.eyebrow{font-family:Arial,sans-serif;font-size:13px;letter-spacing:.18em;text-transform:uppercase;color:var(--accent);margin:0 0 10px}
h1{font-weight:400;font-size:clamp(28px,5vw,48px);line-height:1.1;margin:0 0 18px}
p{font-size:18px;line-height:1.55}.muted{color:var(--muted)}
.fig{background:var(--card);border:1px solid var(--line);padding:10px;margin:18px 0}
.fig img{width:100%;height:auto;display:block}
.cap{font-family:Arial,sans-serif;font-size:13px;color:var(--muted);margin:8px 0 0;line-height:1.45}
.facts{border-top:2px solid var(--accent);margin-top:18px}
.facts div{display:flex;gap:14px;padding:8px 0;border-bottom:1px solid var(--line);font-size:17px}
.facts b{color:var(--accent);font-weight:400;min-width:7.5em}
nav{display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap;font-family:Arial,sans-serif;font-size:14px;margin:28px 0 0;padding-top:14px;border-top:1px solid var(--line)}
a{color:var(--accent)}
.note{font-family:Arial,sans-serif;font-size:12px;color:var(--muted);margin-top:28px}"""
def pagina(r, prev, nxt):
    e=html.escape
    filas="".join(f"<div><b>{k}</b>{e(v)}</div>" for k,v in [("Tamaño",r.tam),("Valor",r.valor),("Permite ver",r.ver),("Fuente",r.fuente)] if v!="—")
    nav_prev=f'<a href="{prev}.html">&larr; {prev}</a>' if prev else '<span></span>'
    nav_next=f'<a href="{nxt}.html">{nxt} &rarr;</a>' if nxt else '<a href="resultados.html">Resultados &rarr;</a>'
    return f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Inventario {r.id} · De Cádiz a Nueva York</title><style>{CSS}</style></head><body><main>
<p class="eyebrow">{e(r.lugar)}</p><h1>{e(r.titulo)}</h1>
<div class="fig"><a href="{r.img}"><img src="{r.img}" alt="{e(r.titulo)}"></a><p class="cap">{e(r.pie)} {e(r.fuente)}.</p></div>
<p>{e(r.texto)}</p><div class="facts">{filas}</div>
<nav>{nav_prev}<a href="index.html">Los siete inventarios</a>{nav_next}</nav>
<p class="note">Lesley Crewdson y José Juan Ruiz · De Cádiz a Nueva York · Universidad de Cádiz, 18 de noviembre de 2026. Toca la imagen para verla a tamaño completo. Actualizado: {date.today():%d-%m-%Y}.</p>
</main></body></html>"""
ids=list(inv["id"])
for k,r in enumerate(inv.itertuples()):
    open(f"{OUT}/{r.id}.html","w",encoding="utf-8").write(pagina(r, ids[k-1] if k>0 else None, ids[k+1] if k<len(ids)-1 else None))
sorted(f for f in os.listdir(OUT) if f.endswith(".html"))

['1805.html',
 '1828.html',
 '1844.html',
 '1867.html',
 '1888.html',
 '1937.html',
 'index.html',
 'resultados.html']

In [4]:
res = f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Resultados · De Cádiz a Nueva York</title><style>{CSS}</style></head><body><main>
<p class="eyebrow">Lo que dicen los inventarios</p><h1>Tres resultados</h1>
<h2 style="font-weight:400">1. La leyenda de «los ingleses»</h2>
<p>Según la tradición familiar, Catalina vendió a los ingleses la mitad de su herencia. Comparada con la de Josefa, la parte de Catalina conserva en 1828 el 84 % de los lotes y el 67 % del valor. Las pérdidas se concentran en Cádiz y en los cuadros más caros: hubo ventas selectivas, no la mitad de la herencia, y lo mejor lo recompraron Josefa y Fernando, empezando por el retrato de Goya.</p>
<h2 style="font-weight:400">2. Meade compró poco, y no lo mejor</h2>
<p>De las 132 obras de los catálogos de Richard W. Meade (1831 y 1853), solo una procede con alta probabilidad de la hijuela de Catalina: un <i>San Pedro liberado de la cárcel</i> atribuido a Caravaggio. Otras diez son posibles.</p>
<h2 style="font-weight:400">3. París no compró</h2>
<p>Al menos 23 de los 51 lotes de Drouot (1867), y 18 de los 19 Murillos, reaparecen en la partición de 1888.</p>
<div class="fig"><img src="img/drouot_1867.png" alt="Lotes de Drouot que reaparecen en 1888"></div>
<div class="fig"><img src="img/pinacoteca_cortes.png" alt="Tamaño de la pinacoteca en cada inventario"><p class="cap">Obras o asientos inventariados, 1805-1937.</p></div>
<nav><a href="1937.html">&larr; 1937</a><a href="index.html">Los siete inventarios</a><a href="../laminas/">Láminas &rarr;</a></nav>
<p class="note">Elaboración propia a partir de las transcripciones de los inventarios. Para estructurar los datos y cruzar los inventarios se ha utilizado Claude (Anthropic), bajo la dirección de los autores, que fijan las ideas centrales, la interpretación y las conclusiones.</p>
</main></body></html>"""
open(f"{OUT}/resultados.html","w",encoding="utf-8").write(res)

3477

In [5]:
def hacer_qr(url, titulo, sub, path):
    q=qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_H, box_size=20, border=4); q.add_data(url); q.make(fit=True)
    img=q.make_image(fill_color="black", back_color="white").convert("RGB"); w,h=img.size
    c=Image.new("RGB",(w,h+150),"white"); c.paste(img,(0,0)); d=ImageDraw.Draw(c)
    try: f1=ImageFont.truetype("DejaVuSans-Bold.ttf",38); f2=ImageFont.truetype("DejaVuSans.ttf",30)
    except OSError: f1=f2=ImageFont.load_default()
    for t,f,y in [(titulo,f1,h),(sub,f2,h+60)]:
        t=t if d.textlength(t,font=f)<w-20 else t[:int(len(t)*(w-40)/d.textlength(t,font=f))]+"…"
        d.text(((w-d.textlength(t,font=f))/2,y),t,fill="black",font=f)
    c.save(path,dpi=(300,300))
for r in inv.itertuples():
    hacer_qr(r.url, f"Inventario {r.id}", r.titulo.split(":")[0], f"{OUT}/qr/QR_inventario_{r.id}.png")
hacer_qr(BASE_URL+"inventarios/resultados.html","Resultados","Lo que dicen los inventarios",f"{OUT}/qr/QR_inventarios_resultados.png")
sorted(os.listdir(f"{OUT}/qr"))

['QR_inventario_1805.png',
 'QR_inventario_1828.png',
 'QR_inventario_1844.png',
 'QR_inventario_1867.png',
 'QR_inventario_1888.png',
 'QR_inventario_1937.png',
 'QR_inventarios.png',
 'QR_inventarios_resultados.png']